# IGDB 003: Taxonomy FK resolution coverage

**FK id → lookup table join rates** for Job 1 / Job 2 artifacts.

Prerequisite: run Job 1 + Job 2:

```bash
python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json
python scripts/recs_job_igdb_games_enriched.py configs/recs_job_igdb_games_enriched.json
```

Steam ↔ IGDB game join rates live in [`igdb_001`](igdb_001_eda_join_coverage.ipynb). Per-field value shapes live in [`igdb_002`](igdb_002_eda_game_review.ipynb).

# Executive Summary

**Question:**  
For each taxonomy FK field on catalog games, what fraction of referenced ids resolve to a row in the matching lookup parquet (`id` → `name` / `name__use`)?

**Result:**  
All five taxonomy fields resolve at **100% global hit rate** (315-game catalog). No missing FK ids in lookup tables. Per-game resolution is perfect wherever a field is populated (0 partial misses). Job 2 enriched parquet parity passes on all 10 `*_names` / `*_names__use` columns.

**Recommendation:**  
No action needed — lookup coverage and Job 2 output are healthy. Re-run this notebook after Job 1/2 artifact changes; investigate only if global hit rate drops below 100% (especially `keywords`, which is `from_games`-scoped).

# Definitions

| Term | Meaning |
|------|--------|
| **Global hit rate** | `|FK ids in games ∩ lookup ids| / |FK ids in games|` (unique ids across catalog) |
| **Per-row hit rate** | For one game, `resolved_names / len(FK id list)`; `None` if FK list empty |
| **Game coverage** | % of catalog rows with non-empty FK list |
| **Resolved coverage** | % of catalog rows with ≥1 resolved name after lookup join |

# Data Sources

| Artifact | Path |
|----------|------|
| Games lookup | `artifacts/igdb/lookups/games.parquet` |
| Taxonomy lookups | `artifacts/igdb/lookups/{genres,themes,...}.parquet` |
| Lookup meta | `artifacts/igdb/lookups/lookup_meta.json` |
| Enriched (parity) | `artifacts/igdb/igdb_games__enriched.parquet` |

# Notebook Roadmap

1. Setup + load lookup parquets
2. Global FK → lookup hit rates (primary diagnostic)
3. Per-game resolution rates + low-coverage samples
4. Missing FK ids (fetch gaps)
5. Enriched parquet parity check (Job 2 smoke test)

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

from steam_review_ml.igdb.constants import (
    IGDB_GAMES_ENRICHED_PARQUET,
    IGDB_GAMES_LOOKUP_PARQUET,
    IGDB_LOOKUP_META_FILENAME,
    IGDB_LOOKUPS_DIR,
    TAXONOMY_FETCH_SCOPE,
    TAXONOMY_RESOLVE_FIELDS,
)
from steam_review_ml.igdb.entity_lookup import (
    _as_id_list,
    build_enriched_games,
    build_lookup_index,
    collect_unique_ids,
    load_taxonomy_lookups,
    resolve_fk_arrays,
    taxonomy_names_column,
    taxonomy_names_embedding_column,
    taxonomy_names_embedding_pooled_column,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
IGDB_DIR = REPO_ROOT / "artifacts/igdb"
LOOKUPS_DIR = IGDB_DIR / IGDB_LOOKUPS_DIR
GAMES_PATH = IGDB_DIR / IGDB_GAMES_LOOKUP_PARQUET
ENRICHED_PATH = IGDB_DIR / IGDB_GAMES_ENRICHED_PARQUET
LOOKUP_META_PATH = IGDB_DIR / IGDB_LOOKUP_META_FILENAME

TAXONOMY_FIELDS = list(TAXONOMY_RESOLVE_FIELDS)
# dont skip and pandas columns
pd.set_option('display.max_columns', None)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"LOOKUPS_DIR={LOOKUPS_DIR}")

REPO_ROOT=/home/ryanr/workspace/steam_recommendations
LOOKUPS_DIR=/home/ryanr/workspace/steam_recommendations/artifacts/igdb/lookups


In [2]:
if not GAMES_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {GAMES_PATH}. Run:\n"
        "  python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json"
    )

games_df = pd.read_parquet(GAMES_PATH)
lookups = load_taxonomy_lookups(LOOKUPS_DIR, TAXONOMY_FIELDS)
lookup_meta = (
    json.loads(LOOKUP_META_PATH.read_text(encoding="utf-8"))
    if LOOKUP_META_PATH.is_file()
    else {}
)

print(f"games rows={len(games_df)}")
print(f"taxonomy fields={TAXONOMY_FIELDS}")
if lookup_meta:
    print(f"lookup_meta entity_row_counts={lookup_meta.get('entity_row_counts')}")

games rows=315
taxonomy fields=['genres', 'themes', 'keywords', 'game_modes', 'player_perspectives']
lookup_meta entity_row_counts={'genres': 23, 'themes': 22, 'keywords': 1707, 'game_modes': 6, 'player_perspectives': 7, 'games': 315}


## Global FK → lookup hit rates

Unique ids referenced on `lookups/games.parquet` vs ids present in each taxonomy lookup table.

In [3]:
def global_fk_stats(games_df: pd.DataFrame, lookup_df: pd.DataFrame, field: str) -> dict[str, Any]:
    lookup_ids = set(lookup_df["id"].astype(int).tolist())
    game_ids = collect_unique_ids(games_df, field)
    hit = game_ids & lookup_ids
    missing = game_ids - lookup_ids
    return {
        "field": field,
        "fetch_scope": TAXONOMY_FETCH_SCOPE.get(field, "from_games"),
        "lookup_rows": len(lookup_df),
        "unique_fk_ids_in_games": len(game_ids),
        "resolved_unique_ids": len(hit),
        "missing_unique_ids": len(missing),
        "global_hit_rate": len(hit) / len(game_ids) if game_ids else np.nan,
        "missing_id_sample": sorted(missing)[:10],
    }


global_overview = pd.DataFrame(
    [global_fk_stats(games_df, lookups[f], f) for f in TAXONOMY_FIELDS]
).set_index("field")

display(
    global_overview[
        [
            "fetch_scope",
            "lookup_rows",
            "unique_fk_ids_in_games",
            "resolved_unique_ids",
            "missing_unique_ids",
            "global_hit_rate",
        ]
    ]
)

,fetch_scope,lookup_rows,unique_fk_ids_in_games,resolved_unique_ids,missing_unique_ids,global_hit_rate
field,,,,,,
genres,full,23,20,20,0,1.0
themes,full,22,22,22,0,1.0
keywords,from_games,1707,1707,1707,0,1.0
game_modes,full,6,6,6,0,1.0
player_perspectives,full,7,7,7,0,1.0


## Per-game resolution rates

In [4]:
def per_game_resolution(games_df: pd.DataFrame, lookup_df: pd.DataFrame, field: str) -> pd.Series:
    lookup_index = build_lookup_index(lookup_df)

    def row_rate(value: Any) -> float | None:
        ids = _as_id_list(value)
        if not ids:
            return None
        names, _ = resolve_fk_arrays(ids, lookup_index)
        return len(names) / len(ids)

    return games_df[field].map(row_rate)


per_game_rows: list[dict[str, Any]] = []
for field in TAXONOMY_FIELDS:
    rates = per_game_resolution(games_df, lookups[field], field)
    populated = games_df[field].map(lambda v: len(_as_id_list(v)) > 0)
    resolved = rates.fillna(0.0) > 0
    per_game_rows.append(
        {
            "field": field,
            "game_coverage_pct": float(populated.mean()) * 100,
            "resolved_coverage_pct": float(resolved.mean()) * 100,
            "per_row_hit_median": float(rates.dropna().median()) if rates.notna().any() else np.nan,
            "per_row_hit_p05": float(rates.dropna().quantile(0.05)) if rates.notna().any() else np.nan,
            "games_with_partial_miss": int(((rates.notna()) & (rates < 1.0)).sum()),
        }
    )

per_game_overview = pd.DataFrame(per_game_rows).set_index("field")
display(per_game_overview)

,game_coverage_pct,resolved_coverage_pct,per_row_hit_median,per_row_hit_p05,games_with_partial_miss
field,,,,,
genres,99.682540,99.682540,1.0,1.0,0
themes,98.095238,98.095238,1.0,1.0,0
keywords,91.111111,91.111111,1.0,1.0,0
game_modes,99.682540,99.682540,1.0,1.0,0
player_perspectives,95.873016,95.873016,1.0,1.0,0


In [5]:
SAMPLE_APP_IDS = [753420, 646910, 512900, 431960]  # incl. Wallpaper Engine mock

recomputed = build_enriched_games(games_df, lookups, TAXONOMY_FIELDS)
sample_cols = ["app_id", "app_name"] + [
    c
    for f in TAXONOMY_FIELDS
    for c in (f, taxonomy_names_column(f))
]
sample_cols = [c for c in sample_cols if c in recomputed.columns]

sample = recomputed[recomputed["app_id"].isin(SAMPLE_APP_IDS)][sample_cols].sort_values("app_id")
display(sample)

,app_id,app_name,genres,genres_names,themes,themes_names,keywords,keywords_names,game_modes,game_modes_names,player_perspectives,player_perspectives_names
314,431960,Wallpaper Engine,[],[],[],[],[],[],[],[],[],[]
2,512900,Streets of Rogue,"[5, 12, 25, 31, 32]","[Shooter, Role-playing (RPG), Hack and slash/B...","[1, 23, 27, 33]","[Action, Stealth, Comedy, Sandbox]","[416, 577, 1033, 1980, 4154, 4466, 4882, 17292...","[roguelike, procedural generation, action-adve...","[1, 2, 3, 4]","[Single player, Multiplayer, Co-operative, Spl...",[3],[Bird view / Isometric]
1,646910,The Crew 2,[10],[Racing],"[1, 38]","[Action, Open world]","[155, 613, 778, 2071, 4357]","[cars, planes, driving, sequel, xbox one x enh...","[1, 2, 3, 5]","[Single player, Multiplayer, Co-operative, Mas...",[2],[Third person]
0,753420,Dungreed,"[8, 31, 32]","[Platform, Adventure, Indie]",[1],[Action],None,[],[1],[Single player],[4],[Side view]


## Missing FK ids

Ids present on games but absent from the lookup table — usually a Job 1 fetch gap (`keywords` is `from_games` scoped).

In [6]:
missing_rows: list[dict[str, Any]] = []
for field in TAXONOMY_FIELDS:
    lookup_ids = set(lookups[field]["id"].astype(int).tolist())
    for entity_id in sorted(collect_unique_ids(games_df, field) - lookup_ids):
        n_games = int(
            games_df[field]
            .map(lambda v: entity_id in _as_id_list(v))
            .sum()
        )
        missing_rows.append({"field": field, "missing_id": entity_id, "n_games": n_games})

missing_df = pd.DataFrame(missing_rows)
if missing_df.empty:
    print("No missing FK ids — all referenced ids resolved in lookup tables.")
else:
    display(missing_df.sort_values(["field", "n_games"], ascending=[True, False]).head(30))

No missing FK ids — all referenced ids resolved in lookup tables.


## Enriched parquet parity (Job 2)

Recompute enriched columns from lookups and compare to saved `igdb_games__enriched.parquet`.

In [7]:
if not ENRICHED_PATH.is_file():
    print(f"Skip parity: missing {ENRICHED_PATH}")
    print("Run: python scripts/recs_job_igdb_games_enriched.py configs/recs_job_igdb_games_enriched.json")
else:
    enriched_saved = pd.read_parquet(ENRICHED_PATH)
    enriched_rebuilt = build_enriched_games(games_df, lookups, TAXONOMY_FIELDS)

    parity_rows: list[dict[str, Any]] = []
    for field in TAXONOMY_FIELDS:
        names_col = taxonomy_names_column(field)
        vecs_col = taxonomy_names_embedding_column(field)
        pooled_col = taxonomy_names_embedding_pooled_column(field)
        for col in (names_col, vecs_col, pooled_col):
            if col not in enriched_saved.columns:
                parity_rows.append({"column": col, "status": "missing_in_saved"})
                continue
            if col.endswith("_names__use_pooled"):
                def _pooled_shape(v: Any) -> int:
                    if isinstance(v, (list, tuple)) and len(v) == 0:
                        return 0
                    return int(np.asarray(v).size)

                saved_shapes = enriched_saved[col].map(_pooled_shape)
                rebuilt_shapes = enriched_rebuilt[col].map(_pooled_shape)
                match = (saved_shapes == rebuilt_shapes).all()
            else:
                saved_lens = enriched_saved[col].map(lambda v: len(v) if isinstance(v, (list, np.ndarray)) else 0)
                rebuilt_lens = enriched_rebuilt[col].map(
                    lambda v: len(v) if isinstance(v, (list, np.ndarray)) else 0
                )
                match = (saved_lens == rebuilt_lens).all()
            parity_rows.append(
                {
                    "column": col,
                    "status": "ok" if match else "length_mismatch",
                    "n_rows": len(enriched_saved),
                }
            )

    parity_df = pd.DataFrame(parity_rows)
    display(parity_df)

    fk_cols = list(TAXONOMY_FIELDS)
    fk_unchanged = enriched_saved[fk_cols].equals(enriched_rebuilt[fk_cols])
    print(f"FK columns unchanged vs games lookup: {fk_unchanged}")

,column,status,n_rows
0,genres_names,ok,315
1,genres_names__use,ok,315
2,genres_names__use_pooled,ok,315
3,themes_names,ok,315
4,themes_names__use,ok,315
5,themes_names__use_pooled,ok,315
6,keywords_names,ok,315
7,keywords_names__use,ok,315
8,keywords_names__use_pooled,ok,315
9,game_modes_names,ok,315


FK columns unchanged vs games lookup: True


In [8]:
enriched_rebuilt.head(5)

,app_id,igdb_game_id,join_method,app_name,age_ratings,franchises,game_engines,game_modes,genres,involved_companies,keywords,multiplayer_modes,igdb_name,player_perspectives,storyline,summary,tags,themes,collections,game_type,summary__use,storyline__use,genres_names,genres_names__use,genres_names__use_pooled,themes_names,themes_names__use,themes_names__use_pooled,keywords_names,keywords_names__use,keywords_names__use_pooled,game_modes_names,game_modes_names__use,game_modes_names__use_pooled,player_perspectives_names,player_perspectives_names__use,player_perspectives_names__use_pooled
0,753420,76816,external_games,Dungreed,"[119531, 55417, 98238, 187890]",None,[13],[1],"[8, 31, 32]","[72008, 128578]",None,None,Dungreed,[4],NaN,Dungreed is 2D side-scrolling action game with...,"[1, 268435464, 268435487, 268435488]",[1],None,0,"[-0.049591586, -0.05844283, 0.015065996, -0.00...","[-0.03564657, -0.033187725, 0.070453554, 0.073...","[Platform, Adventure, Indie]","[[-0.032416463, -0.0116465865, -0.03520837, 0....","[-0.081842445, -0.018193126, 0.03730217, 0.030...",[Action],"[[-0.056234177, -0.008435792, -0.050270304, 0....","[-0.056234177, -0.008435792, -0.050270304, 0.0...",[],[],[],[Single player],"[[-0.03297699, -0.051109567, -0.07943583, 0.01...","[-0.03297699, -0.051109567, -0.07943583, 0.015...",[Side view],"[[-0.04921723, 0.051412594, -0.055959567, 0.06...","[-0.04921723, 0.051412594, -0.055959567, 0.061..."
1,646910,28856,external_games,The Crew 2,"[111035, 111034, 91834, 92244, 215725, 215726,...",None,[118],"[1, 2, 3, 5]",[10],"[50782, 141206, 202299]","[155, 613, 778, 2071, 4357]",[8291],The Crew 2,[2],"The game features a nonlinear story, that foll...",The newest iteration in the revolutionary fran...,"[1, 38, 268435466, 536871067, 536871525, 53687...","[1, 38]",[2719],0,"[-0.015826097, -0.021735007, -0.034556687, -0....","[-0.0007291926, 0.016287263, -0.038463175, -0....",[Racing],"[[-0.010481319, -0.01085115, -0.02164524, 0.04...","[-0.010481319, -0.01085115, -0.02164524, 0.043...","[Action, Open world]","[[-0.056234177, -0.008435792, -0.050270304, 0....","[-0.032937635, -0.021020003, -0.04318108, 0.03...","[cars, planes, driving, sequel, xbox one x enh...","[[-0.056943126, -0.0109716905, 0.041917022, 0....","[-0.046035346, -0.039162543, 0.02567442, 0.037...","[Single player, Multiplayer, Co-operative, Mas...","[[-0.03297699, -0.051109567, -0.07943583, 0.01...","[-0.0099400515, -0.050329078, -0.083285816, 0....",[Third person],"[[-0.015251882, -0.004753101, -0.04244931, -0....","[-0.015251882, -0.004753101, -0.04244931, -0.0..."
2,512900,23275,external_games,Streets of Rogue,"[125538, 195813, 39086, 181520, 57827, 120997]",None,[13],"[1, 2, 3, 4]","[5, 12, 25, 31, 32]","[116692, 116693, 277284]","[416, 577, 1033, 1980, 4154, 4466, 4882, 17292...",[3937],Streets of Rogue,[3],NaN,Streets of Rogue is a top-down rogue-lite with...,"[1, 23, 27, 33, 268435461, 268435468, 26843548...","[1, 23, 27, 33]",[11919],0,"[-0.035389528, -0.053265084, -0.025160592, 0.0...","[-0.03564657, -0.033187725, 0.070453554, 0.073...","[Shooter, Role-playing (RPG), Hack and slash/B...","[[-0.033310566, 0.0396503, -0.009216598, 0.015...","[-0.053734314, -0.0014644319, 0.024830485, -0....","[Action, Stealth, Comedy, Sandbox]","[[-0.056234177, -0.008435792, -0.050270304, 0....","[-0.06844783, 0.021938521, -0.004252563, 0.041...","[roguelike, procedural generation, action-adve...","[[0.009670118, 0.03305227, -0.023673011, -0.05...","[2.8257404e-05, -0.022481047, 0.029408926, -0....","[Single player, Multiplayer, Co-operative, Spl...","[[-0.03297699, -0.051109567, -0.07943583, 0.01...","[-0.0036358987, -0.050688405, -0.079426385, 0....",[Bird view / Isometric],"[[0.011542994, 0.030033382, 0.05908929, 0.0544...","[0.011542994, 0.030033382, 0.05908929, 0.05446..."
3,637090,18279,external_games,BATTLETECH,"[116976, 116977, 116975, 116974]",None,[13],"[1, 2]","[15, 16, 31]","[97027, 97026]","[69, 167, 415, 575, 1107, 1317, 1379, 1448, 24...",None,BattleTech

In [9]:
storyline = enriched_rebuilt.loc[1, "storyline"] if "storyline" in enriched_rebuilt.columns else None
storyline_use = enriched_rebuilt.loc[1, "storyline__use"] if "storyline__use" in enriched_rebuilt.columns else None

print("storyline:", storyline)
print("storyline__use:", storyline_use)
if storyline_use is not None:
    try:
        import numpy as np
        print("storyline__use shape:", np.array(storyline_use).shape)
    except Exception as e:
        print("Unable to determine shape of storyline__use:", e)
else:
    print("storyline__use shape: None")


storyline: The game features a nonlinear story, that follows the unnamed player character as they become a racing icon in the United States by winning in all racing disciplines available in the game. There are four disciplines: Street Racing, Off Road, Freestyle and Pro Racing. In Street Racing, the player is assisted by Latrell Jordan. In Off Road, the player is assisted by Tucker "Tuck" Morgan. In Freestyle, the player is assisted by Sofia Valentina Herrera and her father, Emmett Lee Parker. In Pro Racing, the player is assisted by Alexis Kendrick.
storyline__use: [-0.00072919  0.01628726 -0.03846318 -0.03626097  0.0620758   0.05070741
  0.00538591  0.02443005  0.06459061 -0.05259835 -0.0496914   0.04873702
  0.01317054  0.03609045  0.02504748  0.00020328  0.03197287  0.03569863
 -0.00586857 -0.05253047  0.0130753  -0.05265596  0.00885423 -0.0291941
  0.02491546 -0.01310244  0.00210699 -0.00696142 -0.06600671 -0.02966963
 -0.03783126  0.06437005 -0.03487439 -0.02876892  0.01278275 -0

# Key Findings

**Global FK → lookup hit rates (315 games)**

| Field | Scope | Global hit rate | Missing ids |
|-------|-------|-----------------|-------------|
| genres | full | 100% (20/20) | 0 |
| themes | full | 100% (22/22) | 0 |
| keywords | from_games | 100% (1707/1707) | 0 |
| game_modes | full | 100% (6/6) | 0 |
| player_perspectives | full | 100% (7/7) | 0 |

**Per-game resolution**

- No partial misses on any field (`games_with_partial_miss = 0` everywhere).
- Empty FK lists explain sub-100% *game coverage* (e.g. keywords 91%, player_perspectives 96%) — not lookup gaps.

**Enriched parquet parity (Job 2 smoke test)**

- All 10 resolved columns (`*_names`, `*_names__use`) match rebuilt lengths — **pass**.
- FK columns in enriched parquet unchanged vs `games.parquet` — **pass**.

**Action:** None. Artifacts are consistent; no Job 1 re-fetch required.